# Phase 5 — Feature Engineering

## NexaCart Marketplace Intelligence

This notebook creates business-oriented features from the cleaned datasets to support:

- Exploratory Data Analysis (EDA)
- Statistical Analysis
- Customer Analytics
- Seller Analytics
- Logistics Analysis
- Power BI Dashboard Development
- Business Recommendation Generation

Unlike data cleaning, feature engineering transforms existing data into meaningful business metrics that improve analytical depth and decision-making.

---

### Objectives

- Load cleaned datasets
- Engineer customer, seller, order, product, logistics, payment and review features
- Validate every engineered feature
- Maintain feature engineering logs
- Generate feature summaries
- Export featured datasets

In [1]:
# ==========================================
# Phase 5 : Feature Engineering
# NexaCart Marketplace Intelligence
# ==========================================

# Data Manipulation
import pandas as pd
import numpy as np

# Date & Time
from datetime import datetime

# File Handling
from pathlib import Path

# Display
from IPython.display import display

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# ==========================================
# Project Paths
# ==========================================

PROJECT_ROOT = Path("..")

CLEANED_PATH = PROJECT_ROOT / "data" / "cleaned"
FEATURED_PATH = PROJECT_ROOT / "data" / "featured"
REPORT_PATH = PROJECT_ROOT / "reports"

FEATURED_PATH.mkdir(parents=True, exist_ok=True)
REPORT_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
# ==========================================
# Load Cleaned Data
# ==========================================

customers = pd.read_csv(CLEANED_PATH / "customers.csv")
orders = pd.read_csv(CLEANED_PATH / "orders.csv")
order_items = pd.read_csv(CLEANED_PATH / "order_items.csv")
payments = pd.read_csv(CLEANED_PATH / "payments.csv")
reviews = pd.read_csv(CLEANED_PATH / "reviews.csv")
products = pd.read_csv(CLEANED_PATH / "products.csv")
sellers = pd.read_csv(CLEANED_PATH / "sellers.csv")
geolocation = pd.read_csv(CLEANED_PATH / "geolocation.csv")
category_translation = pd.read_csv(
    CLEANED_PATH / "product_category_translation.csv"
)

In [4]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Category Translation": category_translation
}

overview = []

for name, df in datasets.items():
    overview.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum())
    })

overview_df = pd.DataFrame(overview)

display(overview_df)

,Dataset,Rows,Columns,Missing Values
0,Customers,99441,5,0
1,Orders,99441,8,4908
2,Order Items,112650,7,0
3,Payments,103886,5,0
4,Reviews,99224,5,0
5,Products,32951,9,2448
6,Sellers,3095,4,0
7,Geolocation,738305,5,0
8,Category Translation,71,2,0


In [5]:
# ==========================================
# Feature Engineering Logs
# ==========================================

feature_log = []

feature_summary = []

In [6]:
# ==========================================
# Helper Functions
# ==========================================

def log_feature(dataset, feature_name, description):
    feature_log.append({
        "Dataset": dataset,
        "Feature": feature_name,
        "Description": description
    })


def summarize_feature(df, feature_name):
    feature_summary.append({
        "Feature": feature_name,
        "Datatype": str(df[feature_name].dtype),
        "Missing Values": int(df[feature_name].isna().sum()),
        "Unique Values": int(df[feature_name].nunique())
    })


def validate_feature(df, feature_name):
    print("=" * 60)
    print(f"Validation : {feature_name}")
    print("=" * 60)

    display(df[[feature_name]].head())

    print(df[feature_name].describe(include="all"))

    print("\nMissing Values :", df[feature_name].isna().sum())

    print("-" * 60)

## Feature Engineering Plan

The notebook will engineer features in the following order:

### Section A — Order Features
- Order Processing Time
- Delivery Delay
- Delivery Speed
- Estimated Delivery Difference
- Weekend Order Flag
- Purchase Month
- Purchase Quarter
- Purchase Day
- Purchase Hour
- Purchase Season

---

### Section B — Customer Features
- Customer Lifetime Orders
- Customer Lifetime Spend
- Average Order Value
- Customer Segment
- Repeat Customer
- Customer Recency

---

### Section C — Seller Features
- Seller Total Orders
- Seller Revenue
- Seller Average Rating
- Seller Delivery Performance
- Seller Product Diversity

---

### Section D — Product Features
- Product Volume
- Product Density
- Shipping Efficiency
- Product Category Group

---

### Section E — Payment Features
- Installment Category
- Payment Complexity
- Average Payment per Installment

---

### Section F — Review Features
- Review Sentiment Category
- High Rating Flag
- Low Rating Flag

---

### Section G — Logistics Features
- Freight Percentage
- Delivery Efficiency
- Freight Category

---

Every section will include:

1. Business Objective
2. Feature Creation
3. Validation
4. Summary Table

# Section A — Order Features

## Business Objective

Orders form the backbone of marketplace analytics. By engineering time-based and delivery-related features, we can measure:

- Customer purchasing behavior
- Delivery efficiency
- Logistics performance
- Seasonal trends
- Marketplace growth
- SLA compliance

These features will later power:

- Delivery Performance Dashboard
- Customer Experience Analysis
- Logistics KPIs
- Time Series Analysis
- Seller Performance Evaluation

In [7]:
# ==========================================
# Convert Date Columns
# ==========================================

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("✓ Date columns converted successfully.")

✓ Date columns converted successfully.


In [8]:
orders[date_columns].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [9]:
summary = pd.DataFrame({
    "Column": date_columns,
    "Data Type": [orders[c].dtype for c in date_columns],
    "Missing Values": [orders[c].isna().sum() for c in date_columns]
})

display(summary)

,Column,Data Type,Missing Values
0,order_purchase_timestamp,datetime64[us],0
1,order_approved_at,datetime64[us],160
2,order_delivered_carrier_date,datetime64[us],1783
3,order_delivered_customer_date,datetime64[us],2965
4,order_estimated_delivery_date,datetime64[us],0


## Feature 1 — Order Processing Time

### Business Purpose

Order Processing Time measures how long it takes for an order to move from purchase to approval.

This KPI helps answer:

- How quickly are orders approved?
- Are there operational bottlenecks?
- Which sellers or periods experience slow approval?

### Formula

Order Processing Time (Hours)

= Approval Timestamp − Purchase Timestamp

A lower value indicates faster order processing.

In [10]:
# ==========================================
# Feature 1 : Order Processing Time
# ==========================================

orders["order_processing_hours"] = (
    orders["order_approved_at"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / 3600

log_feature(
    "Orders",
    "order_processing_hours",
    "Hours between purchase and approval"
)

summarize_feature(
    orders,
    "order_processing_hours"
)

In [11]:
validate_feature(
    orders,
    "order_processing_hours"
)

Validation : order_processing_hours


,order_processing_hours
0,0.18
1,30.71
2,0.28
3,0.30
4,1.03


count   99281.00
mean       10.42
std        26.04
min         0.00
25%         0.21
50%         0.34
75%        14.58
max      4509.18
Name: order_processing_hours, dtype: float64

Missing Values : 160
------------------------------------------------------------


In [12]:
processing_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Missing Values"
    ],
    "Value": [
        round(orders["order_processing_hours"].mean(), 2),
        round(orders["order_processing_hours"].median(), 2),
        round(orders["order_processing_hours"].min(), 2),
        round(orders["order_processing_hours"].max(), 2),
        orders["order_processing_hours"].isna().sum()
    ]
})

display(processing_summary)

,Metric,Value
0,Mean,10.42
1,Median,0.34
2,Minimum,0.00
3,Maximum,4509.18
4,Missing Values,160.00


In [13]:
orders["order_processing_hours"].describe(percentiles=[
    .25,.50,.75,.90,.95,.99
])

count   99281.00
mean       10.42
std        26.04
min         0.00
25%         0.21
50%         0.34
75%        14.58
90%        34.66
95%        48.46
99%        90.17
max      4509.18
Name: order_processing_hours, dtype: float64

### Feature 2 — Processing Speed Category

To make processing performance easier to interpret in dashboards, classify orders into operational categories based on approval time.

In [15]:
orders["processing_speed_category"] = pd.Series(index=orders.index, dtype="object")

orders.loc[
    orders["order_processing_hours"] <= 1,
    "processing_speed_category"
] = "Instant (<1 hr)"

orders.loc[
    (orders["order_processing_hours"] > 1) &
    (orders["order_processing_hours"] <= 24),
    "processing_speed_category"
] = "Normal (1-24 hrs)"

orders.loc[
    orders["order_processing_hours"] > 24,
    "processing_speed_category"
] = "Delayed (>24 hrs)"

In [16]:
processing_category_summary = (
    orders["processing_speed_category"]
    .value_counts(dropna=False)
    .rename_axis("Category")
    .reset_index(name="Orders")
)

processing_category_summary["Percentage"] = (
    processing_category_summary["Orders"]
    / len(orders)
    * 100
).round(2)

display(processing_category_summary)

,Category,Orders,Percentage
0,Instant (<1 hr),63425,63.78
1,Normal (1-24 hrs),18437,18.54
2,Delayed (>24 hrs),17419,17.52
3,NaN,160,0.16


## Feature 3 — Delivery Transit Time

### Business Purpose

Delivery Transit Time measures the number of days taken for a shipment to travel from the carrier to the customer.

This KPI helps evaluate:

- Logistics efficiency
- Shipping performance
- Customer delivery experience
- Seller fulfillment effectiveness

### Formula

Delivery Transit Time (Days)

= Customer Delivery Date − Carrier Handover Date

In [17]:
# ==========================================
# Feature 3 : Delivery Transit Time
# ==========================================

orders["delivery_transit_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_delivered_carrier_date"]
).dt.total_seconds() / (24 * 3600)

log_feature(
    "Orders",
    "delivery_transit_days",
    "Days between carrier pickup and customer delivery"
)

summarize_feature(
    orders,
    "delivery_transit_days"
)

In [18]:
validate_feature(
    orders,
    "delivery_transit_days"
)

Validation : delivery_transit_days


,delivery_transit_days
0,6.06
1,12.04
2,9.18
3,9.45
4,1.94


count   96475.00
mean        9.33
std         8.76
min       -16.10
25%         4.10
50%         7.10
75%        12.03
max       205.19
Name: delivery_transit_days, dtype: float64

Missing Values : 2966
------------------------------------------------------------


In [19]:
delivery_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Missing Values"
    ],
    "Value": [
        round(orders["delivery_transit_days"].mean(), 2),
        round(orders["delivery_transit_days"].median(), 2),
        round(orders["delivery_transit_days"].min(), 2),
        round(orders["delivery_transit_days"].max(), 2),
        orders["delivery_transit_days"].isna().sum()
    ]
})

display(delivery_summary)

,Metric,Value
0,Mean,9.33
1,Median,7.10
2,Minimum,-16.10
3,Maximum,205.19
4,Missing Values,2966.00


In [20]:
orders["delivery_transit_days"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count   96475.00
mean        9.33
std         8.76
min       -16.10
25%         4.10
50%         7.10
75%        12.03
90%        18.90
95%        24.21
99%        40.98
max       205.19
Name: delivery_transit_days, dtype: float64

## Feature 4 — Transit Data Quality Flag

Negative delivery durations indicate an impossible logistics sequence where the customer delivery timestamp occurs before the carrier shipment timestamp.

Rather than removing or modifying these records, a validation flag is created to identify them.

This preserves data integrity while allowing downstream analyses to exclude or investigate anomalous records.

In [21]:
# ==========================================
# Feature 4 : Transit Data Quality Flag
# ==========================================

orders["transit_time_valid"] = (
    orders["delivery_transit_days"] >= 0
)

log_feature(
    "Orders",
    "transit_time_valid",
    "Indicates whether delivery transit time is valid (non-negative)"
)

summarize_feature(
    orders,
    "transit_time_valid"
)

In [22]:
validation_summary = (
    orders["transit_time_valid"]
    .value_counts(dropna=False)
    .rename_axis("Valid Transit Time")
    .reset_index(name="Orders")
)

validation_summary["Percentage"] = (
    validation_summary["Orders"] /
    len(orders) * 100
).round(2)

display(validation_summary)

,Valid Transit Time,Orders,Percentage
0,True,96452,96.99
1,False,2989,3.01


In [23]:
orders.loc[
    orders["delivery_transit_days"] < 0,
    [
        "order_id",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "delivery_transit_days"
    ]
].head(10)

,order_id,order_delivered_carrier_date,order_delivered_customer_date,delivery_transit_days
6437,a1abeb653a4d4cd1e142ccb8c82cd069,2017-07-28 16:57:58,2017-07-25 19:32:56,-2.89
9553,383aa8b2724fe452d9ccd9934a8c628b,2017-07-07 17:22:41,2017-07-06 14:27:51,-1.12
13487,cb1134f9010d242e9515ad1c78ec0c39,2017-07-20 19:22:02,2017-07-19 14:13:28,-1.21
14474,dceb62e8fa94b46006c9554fed743df0,2017-08-01 18:23:30,2017-07-26 18:09:10,-6.01
19268,5f9d46795c3126674e52becb3a1a517f,2017-07-20 23:03:42,2017-07-20 18:52:41,-0.17
21338,8c78d01de3a9009e23d6877a7cc9be20,2016-10-26 11:41:53,2016-10-25 17:51:46,-0.74
22520,b27af682321527a6349f1761eb3f360c,2017-06-27 14:51:54,2017-06-26 15:45:35,-0.96
25393,1cc3ae63caffff2d6c3ee3e78e074acf,2017-08-10 18:28:56,2017-08-10 18:05:38,-0.02
25646,e37f11cae9985ca58f0b56f268720537,2017-08-01 18:17:47,2017-07-31 17:49:56,-1.02
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,2017-08-09 18:18:43,2017-08-01 21:13:01,-7.88


In [24]:
# ==========================================
# Feature 4 : Transit Time Status
# ==========================================

orders["transit_time_status"] = np.where(
    orders["delivery_transit_days"].isna(),
    "Not Available",
    np.where(
        orders["delivery_transit_days"] < 0,
        "Invalid",
        "Valid"
    )
)

log_feature(
    "Orders",
    "transit_time_status",
    "Validation status of delivery transit time"
)

summarize_feature(
    orders,
    "transit_time_status"
)

In [25]:
transit_status_summary = (
    orders["transit_time_status"]
    .value_counts()
    .rename_axis("Transit Status")
    .reset_index(name="Orders")
)

transit_status_summary["Percentage"] = (
    transit_status_summary["Orders"] /
    len(orders) * 100
).round(2)

display(transit_status_summary)

,Transit Status,Orders,Percentage
0,Valid,96452,96.99
1,Not Available,2966,2.98
2,Invalid,23,0.02


## Feature 5 — Total Delivery Time

### Business Purpose

Total Delivery Time measures the complete customer waiting period from the moment an order is placed until it is delivered.

Unlike transit time, this metric includes:

- Order approval
- Seller processing
- Carrier handling
- Final delivery

It is a key customer experience KPI.

In [26]:
# ==========================================
# Feature 5 : Total Delivery Time
# ==========================================

orders["total_delivery_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

log_feature(
    "Orders",
    "total_delivery_days",
    "Total days from purchase to customer delivery"
)

summarize_feature(
    orders,
    "total_delivery_days"
)

In [27]:
validate_feature(
    orders,
    "total_delivery_days"
)

Validation : total_delivery_days


,total_delivery_days
0,8.44
1,13.78
2,9.39
3,13.21
4,2.87


count   96476.00
mean       12.56
std         9.55
min         0.53
25%         6.77
50%        10.22
75%        15.72
max       209.63
Name: total_delivery_days, dtype: float64

Missing Values : 2965
------------------------------------------------------------


In [28]:
delivery_time_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Missing Values"
    ],
    "Value": [
        round(orders["total_delivery_days"].mean(), 2),
        round(orders["total_delivery_days"].median(), 2),
        round(orders["total_delivery_days"].min(), 2),
        round(orders["total_delivery_days"].max(), 2),
        orders["total_delivery_days"].isna().sum()
    ]
})

display(delivery_time_summary)

,Metric,Value
0,Mean,12.56
1,Median,10.22
2,Minimum,0.53
3,Maximum,209.63
4,Missing Values,2965.00


## Feature 6 — Delivery Delay

### Business Purpose

Delivery Delay measures the difference between the actual delivery date and the promised delivery date.

This feature is one of the most important logistics KPIs because it directly reflects whether delivery commitments were met.

Interpretation:

- Negative → Delivered before the estimated date
- Zero → Delivered on the estimated date
- Positive → Delivered after the estimated date

In [29]:
# ==========================================
# Feature 6 : Delivery Delay
# ==========================================

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] -
    orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 3600)

log_feature(
    "Orders",
    "delivery_delay_days",
    "Difference between actual and estimated delivery date"
)

summarize_feature(
    orders,
    "delivery_delay_days"
)

In [30]:
validate_feature(
    orders,
    "delivery_delay_days"
)

Validation : delivery_delay_days


,delivery_delay_days
0,-7.11
1,-5.36
2,-17.25
3,-12.98
4,-9.24


count   96476.00
mean      -11.18
std        10.19
min      -146.02
25%       -16.24
50%       -11.95
75%        -6.39
max       188.98
Name: delivery_delay_days, dtype: float64

Missing Values : 2965
------------------------------------------------------------


In [31]:
delay_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Missing Values"
    ],
    "Value": [
        round(orders["delivery_delay_days"].mean(), 2),
        round(orders["delivery_delay_days"].median(), 2),
        round(orders["delivery_delay_days"].min(), 2),
        round(orders["delivery_delay_days"].max(), 2),
        orders["delivery_delay_days"].isna().sum()
    ]
})

display(delay_summary)

,Metric,Value
0,Mean,-11.18
1,Median,-11.95
2,Minimum,-146.02
3,Maximum,188.98
4,Missing Values,2965.00


In [32]:
orders["delivery_status"] = pd.Series(index=orders.index, dtype="object")

orders.loc[
    orders["delivery_delay_days"] < 0,
    "delivery_status"
] = "Early"

orders.loc[
    orders["delivery_delay_days"] == 0,
    "delivery_status"
] = "On Time"

orders.loc[
    orders["delivery_delay_days"] > 0,
    "delivery_status"
] = "Late"

log_feature(
    "Orders",
    "delivery_status",
    "Categorized delivery performance"
)

summarize_feature(
    orders,
    "delivery_status"
)

## Feature 7 — Delivery Status

### Business Purpose

Delivery Status categorizes each delivered order based on whether it arrived before, on, or after the promised delivery date.

This KPI is one of the most important customer experience metrics used in e-commerce.

Categories

- Early
- On Time
- Late

This feature will be used for:

- Seller performance analysis
- Customer satisfaction analysis
- Logistics dashboard
- State-wise delivery performance

In [33]:
# ==========================================
# Feature 7 : Delivery Status
# ==========================================

orders["delivery_status"] = pd.Series(index=orders.index, dtype="object")

actual_date = orders["order_delivered_customer_date"].dt.normalize()
estimated_date = orders["order_estimated_delivery_date"].dt.normalize()

orders.loc[
    actual_date < estimated_date,
    "delivery_status"
] = "Early"

orders.loc[
    actual_date == estimated_date,
    "delivery_status"
] = "On Time"

orders.loc[
    actual_date > estimated_date,
    "delivery_status"
] = "Late"

log_feature(
    "Orders",
    "delivery_status",
    "Delivery performance compared to estimated delivery date"
)

summarize_feature(
    orders,
    "delivery_status"
)

In [34]:
delivery_status_summary = (
    orders["delivery_status"]
    .value_counts(dropna=False)
    .rename_axis("Delivery Status")
    .reset_index(name="Orders")
)

delivery_status_summary["Percentage"] = (
    delivery_status_summary["Orders"] /
    len(orders) * 100
).round(2)

display(delivery_status_summary)

,Delivery Status,Orders,Percentage
0,Early,88649,89.15
1,Late,6535,6.57
2,NaN,2965,2.98
3,On Time,1292,1.30


## Feature 8 — Purchase Year

### Business Purpose

Extract the purchase year to support yearly sales trends and future scalability if new years are added.

In [35]:
# ==========================================
# Feature 8 : Purchase Year
# ==========================================

orders["purchase_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

log_feature(
    "Orders",
    "purchase_year",
    "Year in which the order was placed"
)

summarize_feature(
    orders,
    "purchase_year"
)

## Feature 9 — Purchase Month

### Business Purpose

Extract the purchase month for monthly sales analysis and seasonal demand patterns.

In [36]:
# ==========================================
# Feature 9 : Purchase Month
# ==========================================

orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month
)

orders["purchase_month_name"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

log_feature(
    "Orders",
    "purchase_month",
    "Month number of purchase"
)

log_feature(
    "Orders",
    "purchase_month_name",
    "Month name of purchase"
)

summarize_feature(
    orders,
    "purchase_month"
)

summarize_feature(
    orders,
    "purchase_month_name"
)

In [37]:
# ==========================================
# Feature 10 : Purchase Quarter
# ==========================================

orders["purchase_quarter"] = (
    orders["order_purchase_timestamp"]
    .dt.quarter
)

log_feature(
    "Orders",
    "purchase_quarter",
    "Quarter of purchase"
)

summarize_feature(
    orders,
    "purchase_quarter"
)

In [38]:
# ==========================================
# Feature 11 : Purchase Day
# ==========================================

orders["purchase_day_name"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

orders["purchase_day"] = (
    orders["order_purchase_timestamp"]
    .dt.day
)

log_feature(
    "Orders",
    "purchase_day_name",
    "Day of week of purchase"
)

log_feature(
    "Orders",
    "purchase_day",
    "Day of month"
)

summarize_feature(
    orders,
    "purchase_day_name"
)

summarize_feature(
    orders,
    "purchase_day"
)

In [39]:
# ==========================================
# Feature 12 : Weekend Order
# ==========================================

orders["is_weekend_order"] = (
    orders["order_purchase_timestamp"]
    .dt.dayofweek
    .isin([5, 6])
)

log_feature(
    "Orders",
    "is_weekend_order",
    "Whether the purchase was made on a weekend"
)

summarize_feature(
    orders,
    "is_weekend_order"
)

In [40]:
# ==========================================
# Feature 13 : Purchase Hour
# ==========================================

orders["purchase_hour"] = (
    orders["order_purchase_timestamp"]
    .dt.hour
)

log_feature(
    "Orders",
    "purchase_hour",
    "Hour of purchase"
)

summarize_feature(
    orders,
    "purchase_hour"
)

In [41]:
# ==========================================
# Feature 14 : Time of Day
# ==========================================

orders["time_of_day"] = pd.cut(
    orders["purchase_hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=[
        "Night",
        "Morning",
        "Afternoon",
        "Evening"
    ]
)

log_feature(
    "Orders",
    "time_of_day",
    "Part of the day when the order was placed"
)

summarize_feature(
    orders,
    "time_of_day"
)

In [42]:
section_a_features = [
    "delivery_status",
    "purchase_year",
    "purchase_month",
    "purchase_month_name",
    "purchase_quarter",
    "purchase_day",
    "purchase_day_name",
    "is_weekend_order",
    "purchase_hour",
    "time_of_day"
]

summary = pd.DataFrame({
    "Feature": section_a_features,
    "Datatype": [str(orders[f].dtype) for f in section_a_features],
    "Missing Values": [orders[f].isna().sum() for f in section_a_features],
    "Unique Values": [orders[f].nunique() for f in section_a_features]
})

display(summary)

,Feature,Datatype,Missing Values,Unique Values
0,delivery_status,object,2965,3
1,purchase_year,int32,0,3
2,purchase_month,int32,0,12
3,purchase_month_name,str,0,12
4,purchase_quarter,int32,0,4
5,purchase_day,int32,0,31
6,purchase_day_name,str,0,7
7,is_weekend_order,bool,0,2
8,purchase_hour,int32,0,24
9,time_of_day,category,0,4


# Section B — Customer Features

## Business Objective

Customer-level features summarize purchasing behavior across all orders placed by each customer.

These features enable:

- Customer segmentation
- Lifetime value analysis
- Repeat customer analysis
- Revenue analysis
- RFM Modeling
- Customer Experience Analysis

Unlike the previous section, these features are aggregated at the customer level and stored in the `customers` dataset.

## Customer Feature Preparation

Customer features require information from multiple datasets:

- Customers
- Orders
- Order Items
- Payments
- Reviews

To simplify feature engineering, we first create an Order Master table by combining these datasets.

In [43]:
# ==========================================
# Build Order Master Table
# ==========================================

# Total order value (product price + freight)
order_value = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        total_product_value=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
)

order_value["order_value"] = (
    order_value["total_product_value"] +
    order_value["total_freight"]
)

# Average review score
review_summary = (
    reviews
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean")
    )
)

# Merge
order_master = (
    orders
    .merge(
        customers[
            ["customer_id", "customer_unique_id"]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        order_value,
        on="order_id",
        how="left"
    )
    .merge(
        review_summary,
        on="order_id",
        how="left"
    )
)

print(order_master.shape)

(99441, 30)


In [44]:
order_master.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_processing_hours,processing_speed_category,delivery_transit_days,transit_time_valid,transit_time_status,total_delivery_days,delivery_delay_days,delivery_status,purchase_year,purchase_month,purchase_month_name,purchase_quarter,purchase_day_name,purchase_day,is_weekend_order,purchase_hour,time_of_day,customer_unique_id,total_product_value,total_freight,order_value,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,0.18,Instant (<1 hr),6.06,True,Valid,8.44,-7.11,Early,2017,10,October,4,Monday,2,False,10,Morning,7c396fd4830fd04220f754e42b4e5bff,29.99,8.72,38.71,4.00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,30.71,Delayed (>24 hrs),12.04,True,Valid,13.78,-5.36,Early,2018,7,July,3,Tuesday,24,False,20,Evening,af07308b275d755c9edb36a90c618231,118.70,22.76,141.46,4.00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,0.28,Instant (<1 hr),9.18,True,Valid,9.39,-17.25,Early,2018,8,August,3,Wednesday,8,False,8,Morning,3a653a41f6f9fc3d2a113cf8398680e8,159.90,19.22,179.12,5.00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,0.30,Instant (<1 hr),9.45,True,Valid,13.21,-12.98,Early,2017,11,November,4,Saturday,18,True,19,Evening,7c142cf63193a1473d2e66489a9ae977,45.00,27.20,72.20,5.00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.03,Normal (1-24 hrs),1.94,True,Valid,2.87,-9.24,Early,2018,2,February,1,Tuesday,13,False,21,Evening,72632f0f9dd73dfee390c9b22eb56dd6,19.90,8.72,28.62,5.00


## Feature B1 — Customer Lifetime Orders

### Business Purpose

Customer Lifetime Orders represents the total number of orders placed by each unique customer.

This feature helps identify:

- Repeat customers
- Loyal customers
- High-value customers

It is the foundation for Customer Lifetime Value (CLV) analysis.

In [45]:
# ==========================================
# Feature B1 : Customer Lifetime Orders
# ==========================================

customer_orders = (
    order_master
    .groupby("customer_unique_id")
    .agg(
        lifetime_orders=("order_id", "nunique")
    )
    .reset_index()
)

customers = customers.merge(
    customer_orders,
    on="customer_unique_id",
    how="left"
)

log_feature(
    "Customers",
    "lifetime_orders",
    "Total number of orders placed by customer"
)

summarize_feature(
    customers,
    "lifetime_orders"
)

In [46]:
validate_feature(
    customers,
    "lifetime_orders"
)

Validation : lifetime_orders


,lifetime_orders
0,1
1,1
2,1
3,1
4,1


count   99441.00
mean        1.08
std         0.40
min         1.00
25%         1.00
50%         1.00
75%         1.00
max        17.00
Name: lifetime_orders, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [47]:
customer_order_summary = (
    customers["lifetime_orders"]
    .describe()
    .to_frame()
)

display(customer_order_summary)

,lifetime_orders
count,99441.00
mean,1.08
std,0.40
min,1.00
25%,1.00
50%,1.00
75%,1.00
max,17.00


In [48]:
# ==========================================
# Feature B2 : Repeat Customer
# ==========================================

customers["is_repeat_customer"] = (
    customers["lifetime_orders"] > 1
)

log_feature(
    "Customers",
    "is_repeat_customer",
    "Whether customer has placed more than one order"
)

summarize_feature(
    customers,
    "is_repeat_customer"
)

In [49]:
repeat_summary = (
    customers["is_repeat_customer"]
    .value_counts()
    .rename_axis("Repeat Customer")
    .reset_index(name="Customers")
)

repeat_summary["Percentage"] = (
    repeat_summary["Customers"] /
    len(customers) * 100
).round(2)

display(repeat_summary)

,Repeat Customer,Customers,Percentage
0,False,93099,93.62
1,True,6342,6.38


## Feature B3 — Customer Lifetime Spend

### Business Purpose

Customer Lifetime Spend represents the total revenue generated by each customer across all completed orders.

This feature helps identify:

- High-value customers
- Customer purchasing power
- Revenue concentration
- Customer Lifetime Value (CLV)

It is a foundational metric for customer segmentation and revenue analysis.

In [50]:
# ==========================================
# Feature B3 : Customer Lifetime Spend
# ==========================================

customer_spend = (
    order_master
    .groupby("customer_unique_id", as_index=False)
    .agg(
        lifetime_spend=("order_value", "sum")
    )
)

customers = customers.merge(
    customer_spend,
    on="customer_unique_id",
    how="left"
)

log_feature(
    "Customers",
    "lifetime_spend",
    "Total amount spent by customer across all orders"
)

summarize_feature(
    customers,
    "lifetime_spend"
)

In [51]:
validate_feature(
    customers,
    "lifetime_spend"
)

Validation : lifetime_spend


,lifetime_spend
0,146.87
1,335.48
2,157.73
3,173.30
4,252.25


count   99441.00
mean      170.65
std       235.78
min         0.00
25%        63.37
50%       110.20
75%       188.65
max     13664.08
Name: lifetime_spend, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [52]:
customer_spend_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Missing Values"
    ],
    "Value": [
        round(customers["lifetime_spend"].mean(), 2),
        round(customers["lifetime_spend"].median(), 2),
        round(customers["lifetime_spend"].min(), 2),
        round(customers["lifetime_spend"].max(), 2),
        customers["lifetime_spend"].isna().sum()
    ]
})

display(customer_spend_summary)

,Metric,Value
0,Mean,170.65
1,Median,110.20
2,Minimum,0.00
3,Maximum,13664.08
4,Missing Values,0.00


## Feature B4 — Average Order Value (AOV)

### Business Purpose

Average Order Value (AOV) measures the average amount spent per order by each customer.

Formula:

Average Order Value =
Customer Lifetime Spend ÷ Customer Lifetime Orders

This KPI helps identify customers who make higher-value purchases.

In [53]:
# ==========================================
# Feature B4 : Average Order Value
# ==========================================

customers["average_order_value"] = (
    customers["lifetime_spend"] /
    customers["lifetime_orders"]
).round(2)

log_feature(
    "Customers",
    "average_order_value",
    "Average spend per order"
)

summarize_feature(
    customers,
    "average_order_value"
)

In [54]:
validate_feature(
    customers,
    "average_order_value"
)

Validation : average_order_value


,average_order_value
0,146.87
1,335.48
2,157.73
3,173.30
4,252.25


count   99441.00
mean      159.33
std       218.58
min         0.00
25%        61.85
50%       105.28
75%       175.99
max     13664.08
Name: average_order_value, dtype: float64

Missing Values : 0
------------------------------------------------------------


## Feature B5 — Customer Recency

### Business Purpose

Customer Recency measures the number of days since a customer's most recent purchase.

It is calculated relative to the latest purchase date in the dataset.

A lower value indicates a more recently active customer.

This feature is a core component of RFM (Recency, Frequency, Monetary) analysis.

In [55]:
# ==========================================
# Feature B5 : Customer Recency
# ==========================================

# Reference date (latest purchase in the dataset)
reference_date = order_master["order_purchase_timestamp"].max()

customer_recency = (
    order_master
    .groupby("customer_unique_id", as_index=False)
    .agg(
        last_purchase=("order_purchase_timestamp", "max")
    )
)

customer_recency["customer_recency_days"] = (
    reference_date -
    customer_recency["last_purchase"]
).dt.days

customers = customers.merge(
    customer_recency[
        ["customer_unique_id", "customer_recency_days"]
    ],
    on="customer_unique_id",
    how="left"
)

log_feature(
    "Customers",
    "customer_recency_days",
    "Days since customer's most recent purchase"
)

summarize_feature(
    customers,
    "customer_recency_days"
)

In [56]:
validate_feature(
    customers,
    "customer_recency_days"
)

Validation : customer_recency_days


,customer_recency_days
0,519
1,277
2,151
3,218
4,80


count   99441.00
mean      286.95
std       153.24
min         0.00
25%       163.00
50%       268.00
75%       396.00
max       772.00
Name: customer_recency_days, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [57]:
recency_summary = pd.DataFrame({
    "Metric": [
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Missing Values"
    ],
    "Value": [
        round(customers["customer_recency_days"].mean(), 2),
        round(customers["customer_recency_days"].median(), 2),
        round(customers["customer_recency_days"].min(), 2),
        round(customers["customer_recency_days"].max(), 2),
        customers["customer_recency_days"].isna().sum()
    ]
})

display(recency_summary)

,Metric,Value
0,Mean,286.95
1,Median,268.00
2,Minimum,0.00
3,Maximum,772.00
4,Missing Values,0.00


In [58]:
customer_features = [
    "lifetime_orders",
    "is_repeat_customer",
    "lifetime_spend",
    "average_order_value",
    "customer_recency_days"
]

customer_summary = pd.DataFrame({
    "Feature": customer_features,
    "Datatype": [str(customers[f].dtype) for f in customer_features],
    "Missing Values": [customers[f].isna().sum() for f in customer_features],
    "Unique Values": [customers[f].nunique() for f in customer_features]
})

display(customer_summary)

,Feature,Datatype,Missing Values,Unique Values
0,lifetime_orders,int64,0,9
1,is_repeat_customer,bool,0,2
2,lifetime_spend,float64,0,31718
3,average_order_value,float64,0,27213
4,customer_recency_days,int64,0,630


## Feature B6 — Purchase Frequency

### Business Purpose

Purchase Frequency measures how frequently a customer places orders over their customer lifetime.

Unlike Lifetime Orders, Purchase Frequency considers the time span between the customer's first and last purchase.

Formula

Purchase Frequency =
Lifetime Orders ÷ Customer Lifetime (Days)

Higher values indicate more frequent purchasing behavior.

In [59]:
# ==========================================
# Customer Lifetime (Days)
# ==========================================

customer_dates = (
    order_master
    .groupby("customer_unique_id")
    .agg(
        first_purchase=("order_purchase_timestamp","min"),
        last_purchase=("order_purchase_timestamp","max")
    )
    .reset_index()
)

customer_dates["customer_lifetime_days"] = (
    customer_dates["last_purchase"] -
    customer_dates["first_purchase"]
).dt.days

In [60]:
customer_dates["customer_lifetime_days"] = (
    customer_dates["customer_lifetime_days"]
    .replace(0,1)
)

In [61]:
customers = customers.merge(
    customer_dates[
        [
            "customer_unique_id",
            "customer_lifetime_days"
        ]
    ],
    on="customer_unique_id",
    how="left"
)

In [62]:
# ==========================================
# Feature B6 : Purchase Frequency
# ==========================================

customers["purchase_frequency"] = (
    customers["lifetime_orders"] /
    customers["customer_lifetime_days"]
).round(4)

log_feature(
    "Customers",
    "purchase_frequency",
    "Average orders per customer lifetime day"
)

summarize_feature(
    customers,
    "purchase_frequency"
)

In [63]:
validate_feature(
    customers,
    "purchase_frequency"
)

Validation : purchase_frequency


,purchase_frequency
0,1.00
1,1.00
2,1.00
3,1.00
4,1.00


count   99441.00
mean        0.98
std         0.25
min         0.00
25%         1.00
50%         1.00
75%         1.00
max         6.00
Name: purchase_frequency, dtype: float64

Missing Values : 0
------------------------------------------------------------


## Feature B7 — Customer Average Review Score

### Business Purpose

Measures the average review score given by a customer across all completed purchases.

Useful for:

- Customer satisfaction analysis
- Customer segmentation
- Loyalty studies

In [64]:
customer_review = (
    order_master
    .groupby("customer_unique_id")
    .agg(
        customer_avg_review=("review_score","mean")
    )
    .reset_index()
)

customers = customers.merge(
    customer_review,
    on="customer_unique_id",
    how="left"
)

log_feature(
    "Customers",
    "customer_avg_review",
    "Average review score given by customer"
)

summarize_feature(
    customers,
    "customer_avg_review"
)

In [65]:
validate_feature(
    customers,
    "customer_avg_review"
)

Validation : customer_avg_review


,customer_avg_review
0,4.00
1,5.00
2,5.00
3,5.00
4,5.00


count   98716.00
mean        4.09
std         1.34
min         1.00
25%         4.00
50%         5.00
75%         5.00
max         5.00
Name: customer_avg_review, dtype: float64

Missing Values : 725
------------------------------------------------------------


In [66]:
customer_delivery = (
    order_master
    .groupby("customer_unique_id")
    .agg(
        customer_avg_delivery_days=("total_delivery_days","mean")
    )
    .reset_index()
)

customers = customers.merge(
    customer_delivery,
    on="customer_unique_id",
    how="left"
)

log_feature(
    "Customers",
    "customer_avg_delivery_days",
    "Average delivery time experienced by customer"
)

summarize_feature(
    customers,
    "customer_avg_delivery_days"
)

In [67]:
validate_feature(
    customers,
    "customer_avg_delivery_days"
)

Validation : customer_avg_delivery_days


,customer_avg_delivery_days
0,8.81
1,16.66
2,26.08
3,15.00
4,11.46


count   96681.00
mean       12.56
std         9.47
min         0.53
25%         6.81
50%        10.26
75%        15.69
max       209.63
Name: customer_avg_delivery_days, dtype: float64

Missing Values : 2760
------------------------------------------------------------


In [68]:
customer_processing = (
    order_master
    .groupby("customer_unique_id")
    .agg(
        customer_avg_processing_hours=(
            "order_processing_hours",
            "mean"
        )
    )
    .reset_index()
)

customers = customers.merge(
    customer_processing,
    on="customer_unique_id",
    how="left"
)

log_feature(
    "Customers",
    "customer_avg_processing_hours",
    "Average order processing time"
)

summarize_feature(
    customers,
    "customer_avg_processing_hours"
)

In [69]:
validate_feature(
    customers,
    "customer_avg_processing_hours"
)

Validation : customer_avg_processing_hours


,customer_avg_processing_hours
0,0.28
1,0.17
2,24.19
3,1.38
4,0.31


count   99338.00
mean       10.42
std        25.89
min         0.00
25%         0.22
50%         0.35
75%        14.65
max      4509.18
Name: customer_avg_processing_hours, dtype: float64

Missing Values : 103
------------------------------------------------------------


## Feature B10 — Customer Value Tier

### Business Purpose

Customers are segmented into value tiers based on Lifetime Spend.

The segmentation uses quartiles to provide a balanced distribution of customers across value levels.

Value Tiers

- Low Value
- Medium Value
- High Value
- Premium

In [70]:
customers["customer_value_tier"] = pd.qcut(
    customers["lifetime_spend"],
    q=4,
    labels=[
        "Low Value",
        "Medium Value",
        "High Value",
        "Premium"
    ]
)

log_feature(
    "Customers",
    "customer_value_tier",
    "Customer segmentation based on lifetime spend"
)

summarize_feature(
    customers,
    "customer_value_tier"
)

In [71]:
value_summary = (
    customers["customer_value_tier"]
    .value_counts()
    .rename_axis("Customer Tier")
    .reset_index(name="Customers")
)

value_summary["Percentage"] = (
    value_summary["Customers"] /
    len(customers)
    *100
).round(2)

display(value_summary)

,Customer Tier,Customers,Percentage
0,Low Value,24865,25.00
1,Medium Value,24860,25.00
2,Premium,24859,25.00
3,High Value,24857,25.00


In [72]:
customer_features = [
    "lifetime_orders",
    "is_repeat_customer",
    "lifetime_spend",
    "average_order_value",
    "customer_recency_days",
    "customer_lifetime_days",
    "purchase_frequency",
    "customer_avg_review",
    "customer_avg_delivery_days",
    "customer_avg_processing_hours",
    "customer_value_tier"
]

summary = pd.DataFrame({
    "Feature": customer_features,
    "Datatype": [str(customers[f].dtype) for f in customer_features],
    "Missing Values": [customers[f].isna().sum() for f in customer_features],
    "Unique Values": [customers[f].nunique() for f in customer_features]
})

display(summary)

,Feature,Datatype,Missing Values,Unique Values
0,lifetime_orders,int64,0,9
1,is_repeat_customer,bool,0,2
2,lifetime_spend,float64,0,31718
3,average_order_value,float64,0,27213
4,customer_recency_days,int64,0,630
5,customer_lifetime_days,int64,0,421
6,purchase_frequency,float64,0,314
7,customer_avg_review,float64,725,34
8,customer_avg_delivery_days,float64,2760,90947
9,customer_avg_processing_hours,float64,103,33838


# Section C — Seller Features

## Business Objective

Seller performance directly influences customer satisfaction, delivery efficiency, and marketplace revenue.

This section engineers seller-level business metrics by aggregating order, product, payment, logistics, and review information.

The engineered features will support:

- Seller performance evaluation
- Revenue analysis
- Logistics analysis
- Customer satisfaction analysis
- Seller segmentation
- Power BI dashboards

In [73]:
# ==========================================
# Seller Master Table
# ==========================================

seller_master = (
    order_items.merge(
        orders,
        on="order_id",
        how="left"
    )
    .merge(
        reviews[["order_id", "review_score"]],
        on="order_id",
        how="left"
    )
)

print(seller_master.shape)

seller_master.head()

(113314, 32)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_processing_hours,processing_speed_category,delivery_transit_days,transit_time_valid,transit_time_status,total_delivery_days,delivery_delay_days,delivery_status,purchase_year,purchase_month,purchase_month_name,purchase_quarter,purchase_day_name,purchase_day,is_weekend_order,purchase_hour,time_of_day,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,0.78,Instant (<1 hr),1.21,True,Valid,7.61,-8.01,Early,2017,9,September,3,Wednesday,13,False,8,Morning,5.00
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,0.20,Instant (<1 hr),8.06,True,Valid,16.22,-2.33,Early,2017,4,April,2,Wednesday,26,False,10,Morning,4.00
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,0.25,Instant (<1 hr),6.03,True,Valid,7.95,-13.44,Early,2018,1,January,1,Sunday,14,True,14,Afternoon,5.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,0.16,Instant (<1 hr),4.00,True,Valid,6.15,-5.44,Early,2018,8,August,3,Wednesday,8,False,10,Morning,4.00
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,0.21,Instant (<1 hr),13.29,True,Valid,25.11,-15.30,Early,2017,2,February,1,Saturday,4,True,13,Afternoon,5.00


In [74]:
seller_orders = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_total_orders=("order_id", "nunique")
    )
)

sellers = sellers.merge(
    seller_orders,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_total_orders",
    "Total unique orders fulfilled by seller"
)

summarize_feature(
    sellers,
    "seller_total_orders"
)

validate_feature(
    sellers,
    "seller_total_orders"
)

Validation : seller_total_orders


,seller_total_orders
0,3
1,40
2,1
3,1
4,1


count   3095.00
mean      32.31
std      105.14
min        1.00
25%        2.00
50%        6.00
75%       21.50
max     1854.00
Name: seller_total_orders, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [75]:
seller_revenue = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        seller_revenue=("price", "sum")
    )
)

sellers = sellers.merge(
    seller_revenue,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_revenue",
    "Total revenue generated by seller"
)

summarize_feature(
    sellers,
    "seller_revenue"
)

validate_feature(
    sellers,
    "seller_revenue"
)

Validation : seller_revenue


,seller_revenue
0,218.70
1,11703.07
2,158.00
3,79.99
4,167.99


count     3095.00
mean      4391.48
std      13922.00
min          3.50
25%        208.85
50%        821.48
75%       3280.83
max     229472.63
Name: seller_revenue, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [76]:
seller_aov = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_avg_order_value=("price", "mean")
    )
)

sellers = sellers.merge(
    seller_aov,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_avg_order_value",
    "Average order value per seller"
)

summarize_feature(
    sellers,
    "seller_avg_order_value"
)

validate_feature(
    sellers,
    "seller_avg_order_value"
)

Validation : seller_avg_order_value


,seller_avg_order_value
0,72.90
1,285.44
2,158.00
3,79.99
4,167.99


count   3095.00
mean     176.26
std      322.10
min        3.50
25%       52.09
50%       95.40
75%      174.20
max     6729.00
Name: seller_avg_order_value, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [77]:
seller_reviews = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_avg_review=("review_score", "mean")
    )
)

sellers = sellers.merge(
    seller_reviews,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_avg_review",
    "Average customer review score"
)

summarize_feature(
    sellers,
    "seller_avg_review"
)

validate_feature(
    sellers,
    "seller_avg_review"
)

Validation : seller_avg_review


,seller_avg_review
0,3.00
1,4.56
2,5.00
3,5.00
4,1.00


count   3090.00
mean       3.97
std        0.97
min        1.00
25%        3.71
50%        4.17
75%        4.60
max        5.00
Name: seller_avg_review, dtype: float64

Missing Values : 5
------------------------------------------------------------


In [78]:
seller_delivery = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_avg_delivery_days=("total_delivery_days", "mean")
    )
)

sellers = sellers.merge(
    seller_delivery,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_avg_delivery_days",
    "Average customer delivery time"
)

summarize_feature(
    sellers,
    "seller_avg_delivery_days"
)

validate_feature(
    sellers,
    "seller_avg_delivery_days"
)

Validation : seller_avg_delivery_days


,seller_avg_delivery_days
0,13.02
1,9.07
2,4.04
3,5.67
4,35.31


count   2970.00
mean      12.17
std        7.10
min        1.21
25%        8.28
50%       11.14
75%       14.23
max      189.86
Name: seller_avg_delivery_days, dtype: float64

Missing Values : 125
------------------------------------------------------------


In [79]:
seller_processing = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_avg_processing_hours=(
            "order_processing_hours",
            "mean"
        )
    )
)

sellers = sellers.merge(
    seller_processing,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_avg_processing_hours",
    "Average order approval time"
)

summarize_feature(
    sellers,
    "seller_avg_processing_hours"
)

validate_feature(
    sellers,
    "seller_avg_processing_hours"
)

Validation : seller_avg_processing_hours


,seller_avg_processing_hours
0,0.50
1,11.61
2,0.41
3,0.30
4,0.19


count   3095.00
mean      10.42
std       11.70
min        0.00
25%        2.33
50%        8.47
75%       13.56
max      139.12
Name: seller_avg_processing_hours, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [80]:
seller_products = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        seller_product_diversity=("product_id", "nunique")
    )
)

sellers = sellers.merge(
    seller_products,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_product_diversity",
    "Number of unique products sold"
)

summarize_feature(
    sellers,
    "seller_product_diversity"
)

In [81]:
seller_order_values = (
    order_items
    .groupby(
        ["seller_id", "order_id"],
        as_index=False
    )
    .agg(
        order_total=("price","sum")
    )
)

seller_aov = (
    seller_order_values
    .groupby("seller_id",as_index=False)
    .agg(
        seller_avg_order_value=("order_total","mean")
    )
)

In [82]:
# ==========================================
# Feature C8 : Seller Average Freight
# ==========================================

seller_freight = (
    order_items
    .groupby("seller_id", as_index=False)
    .agg(
        seller_avg_freight=("freight_value", "mean")
    )
)

sellers = sellers.merge(
    seller_freight,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_avg_freight",
    "Average freight charged by seller"
)

summarize_feature(
    sellers,
    "seller_avg_freight"
)

validate_feature(
    sellers,
    "seller_avg_freight"
)

Validation : seller_avg_freight


,seller_avg_freight
0,9.30
1,35.09
2,16.21
3,15.66
4,31.93


count   3095.00
mean      23.38
std       18.96
min        1.20
25%       14.74
50%       18.23
75%       24.37
max      308.34
Name: seller_avg_freight, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [83]:
# ==========================================
# Feature C9 : Seller Customer Count
# ==========================================

seller_customers = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_customer_count=("customer_id", "nunique")
    )
)

sellers = sellers.merge(
    seller_customers,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_customer_count",
    "Unique customers served by seller"
)

summarize_feature(
    sellers,
    "seller_customer_count"
)

validate_feature(
    sellers,
    "seller_customer_count"
)

Validation : seller_customer_count


,seller_customer_count
0,3
1,40
2,1
3,1
4,1


count   3095.00
mean      32.31
std      105.14
min        1.00
25%        2.00
50%        6.00
75%       21.50
max     1854.00
Name: seller_customer_count, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [84]:
# ==========================================
# Feature C10 : Seller Late Delivery Rate
# ==========================================

late_orders = (
    seller_master
    .assign(
        is_late=seller_master["delivery_status"] == "Late"
    )
    .groupby("seller_id", as_index=False)
    .agg(
        seller_late_delivery_rate=("is_late", "mean")
    )
)

late_orders["seller_late_delivery_rate"] *= 100

sellers = sellers.merge(
    late_orders,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_late_delivery_rate",
    "Percentage of late deliveries"
)

summarize_feature(
    sellers,
    "seller_late_delivery_rate"
)

validate_feature(
    sellers,
    "seller_late_delivery_rate"
)

Validation : seller_late_delivery_rate


,seller_late_delivery_rate
0,33.33
1,2.44
2,0.00
3,0.00
4,100.00


count   3095.00
mean       6.22
std       14.45
min        0.00
25%        0.00
50%        0.00
75%        6.96
max      100.00
Name: seller_late_delivery_rate, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [85]:
# ==========================================
# Feature C11 : Seller Early Delivery Rate
# ==========================================

early_orders = (
    seller_master
    .assign(
        is_early=seller_master["delivery_status"] == "Early"
    )
    .groupby("seller_id", as_index=False)
    .agg(
        seller_early_delivery_rate=("is_early", "mean")
    )
)

early_orders["seller_early_delivery_rate"] *= 100

sellers = sellers.merge(
    early_orders,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_early_delivery_rate",
    "Percentage of early deliveries"
)

summarize_feature(
    sellers,
    "seller_early_delivery_rate"
)

In [86]:
# ==========================================
# Feature C12 : Seller Revenue Rank
# ==========================================

sellers["seller_revenue_rank"] = (
    sellers["seller_revenue"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

log_feature(
    "Sellers",
    "seller_revenue_rank",
    "Revenue ranking of seller"
)

summarize_feature(
    sellers,
    "seller_revenue_rank"
)

In [87]:
# ==========================================
# Feature C13 : Seller Performance Tier
# ==========================================

sellers["seller_performance_tier"] = pd.qcut(
    sellers["seller_revenue"],
    q=4,
    labels=[
        "Bronze",
        "Silver",
        "Gold",
        "Platinum"
    ]
)

log_feature(
    "Sellers",
    "seller_performance_tier",
    "Seller performance based on revenue quartiles"
)

summarize_feature(
    sellers,
    "seller_performance_tier"
)

In [88]:
seller_features = [
    "seller_total_orders",
    "seller_revenue",
    "seller_avg_order_value",
    "seller_avg_review",
    "seller_avg_delivery_days",
    "seller_avg_processing_hours",
    "seller_product_diversity",
    "seller_avg_freight",
    "seller_customer_count",
    "seller_late_delivery_rate",
    "seller_early_delivery_rate",
    "seller_revenue_rank",
    "seller_performance_tier"
]

seller_summary = pd.DataFrame({
    "Feature": seller_features,
    "Datatype": [str(sellers[f].dtype) for f in seller_features],
    "Missing Values": [sellers[f].isna().sum() for f in seller_features],
    "Unique Values": [sellers[f].nunique() for f in seller_features]
})

display(seller_summary)

,Feature,Datatype,Missing Values,Unique Values
0,seller_total_orders,int64,0,251
1,seller_revenue,float64,0,2778
2,seller_avg_order_value,float64,0,2685
3,seller_avg_review,float64,5,815
4,seller_avg_delivery_days,float64,125,2970
5,seller_avg_processing_hours,float64,0,3024
6,seller_product_diversity,int64,0,130
7,seller_avg_freight,float64,0,2868
8,seller_customer_count,int64,0,251
9,seller_late_delivery_rate,float64,0,446


# Section D — Product Features

## Business Objective

Product-level features enrich the product dataset with business metrics derived from sales, logistics, and customer feedback.

These engineered features enable:

- Product performance analysis
- Revenue analysis
- Customer preference analysis
- Logistics optimization
- Product segmentation

The resulting dataset will support Exploratory Data Analysis (EDA) and Power BI dashboards.

In [89]:
# ==========================================
# Build Product Master
# ==========================================

product_master = (
    order_items
    .merge(
        orders,
        on="order_id",
        how="left"
    )
    .merge(
        reviews[["order_id", "review_score"]],
        on="order_id",
        how="left"
    )
)

print(product_master.shape)

product_master.head()

(113314, 32)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_processing_hours,processing_speed_category,delivery_transit_days,transit_time_valid,transit_time_status,total_delivery_days,delivery_delay_days,delivery_status,purchase_year,purchase_month,purchase_month_name,purchase_quarter,purchase_day_name,purchase_day,is_weekend_order,purchase_hour,time_of_day,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,0.78,Instant (<1 hr),1.21,True,Valid,7.61,-8.01,Early,2017,9,September,3,Wednesday,13,False,8,Morning,5.00
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,0.20,Instant (<1 hr),8.06,True,Valid,16.22,-2.33,Early,2017,4,April,2,Wednesday,26,False,10,Morning,4.00
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,0.25,Instant (<1 hr),6.03,True,Valid,7.95,-13.44,Early,2018,1,January,1,Sunday,14,True,14,Afternoon,5.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,0.16,Instant (<1 hr),4.00,True,Valid,6.15,-5.44,Early,2018,8,August,3,Wednesday,8,False,10,Morning,4.00
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,0.21,Instant (<1 hr),13.29,True,Valid,25.11,-15.30,Early,2017,2,February,1,Saturday,4,True,13,Afternoon,5.00


## Feature D1 — Product Sales Count

### Business Purpose

Measures how many times each product has been sold.

Higher values indicate more popular products.

In [90]:
# ==========================================
# Feature D1 : Product Sales Count
# ==========================================

product_sales = (
    product_master
    .groupby("product_id", as_index=False)
    .agg(
        product_sales_count=("order_id", "count")
    )
)

products = products.merge(
    product_sales,
    on="product_id",
    how="left"
)

products["product_sales_count"] = (
    products["product_sales_count"]
    .fillna(0)
    .astype(int)
)

log_feature(
    "Products",
    "product_sales_count",
    "Total number of times the product was sold"
)

summarize_feature(
    products,
    "product_sales_count"
)

validate_feature(
    products,
    "product_sales_count"
)

Validation : product_sales_count


,product_sales_count
0,1
1,1
2,1
3,1
4,1


count   32951.00
mean        3.44
std        10.69
min         1.00
25%         1.00
50%         1.00
75%         3.00
max       527.00
Name: product_sales_count, dtype: float64

Missing Values : 0
------------------------------------------------------------


## Feature D2 — Product Revenue

### Business Purpose

Represents the total revenue generated by each product across all orders.

This feature is essential for identifying the highest revenue-generating products.

In [91]:
# ==========================================
# Feature D2 : Product Revenue
# ==========================================

product_revenue = (
    order_items
    .groupby("product_id", as_index=False)
    .agg(
        product_revenue=("price", "sum")
    )
)

products = products.merge(
    product_revenue,
    on="product_id",
    how="left"
)

products["product_revenue"] = (
    products["product_revenue"]
    .fillna(0)
)

log_feature(
    "Products",
    "product_revenue",
    "Total revenue generated by product"
)

summarize_feature(
    products,
    "product_revenue"
)

validate_feature(
    products,
    "product_revenue"
)

Validation : product_revenue


,product_revenue
0,10.91
1,248.00
2,79.80
3,112.30
4,37.90


count   32951.00
mean      412.48
std      1371.95
min         2.20
25%        59.90
50%       136.75
75%       329.00
max     63885.00
Name: product_revenue, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [92]:
# ==========================================
# Feature D3 : Average Selling Price
# ==========================================

product_price = (
    order_items
    .groupby("product_id", as_index=False)
    .agg(
        average_selling_price=("price", "mean")
    )
)

products = products.merge(
    product_price,
    on="product_id",
    how="left"
)

log_feature(
    "Products",
    "average_selling_price",
    "Average selling price of the product"
)

summarize_feature(
    products,
    "average_selling_price"
)

validate_feature(
    products,
    "average_selling_price"
)

Validation : average_selling_price


,average_selling_price
0,10.91
1,248.00
2,79.80
3,112.30
4,37.90


count   32951.00
mean      145.30
std       246.90
min         0.85
25%        39.90
50%        79.00
75%       154.90
max      6735.00
Name: average_selling_price, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [93]:
# ==========================================
# Feature D4 : Average Freight
# ==========================================

product_freight = (
    order_items
    .groupby("product_id", as_index=False)
    .agg(
        average_freight=("freight_value", "mean")
    )
)

products = products.merge(
    product_freight,
    on="product_id",
    how="left"
)

log_feature(
    "Products",
    "average_freight",
    "Average freight charged for the product"
)

summarize_feature(
    products,
    "average_freight"
)

validate_feature(
    products,
    "average_freight"
)

Validation : average_freight


,average_freight
0,7.39
1,17.99
2,7.82
3,9.54
4,8.29


count   32951.00
mean       21.20
std        18.09
min         0.01
25%        13.60
50%        16.70
75%        21.83
max       409.68
Name: average_freight, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [94]:
# ==========================================
# Feature D5 : Product Average Rating
# ==========================================

product_rating = (
    product_master
    .groupby("product_id", as_index=False)
    .agg(
        average_product_rating=("review_score", "mean")
    )
)

products = products.merge(
    product_rating,
    on="product_id",
    how="left"
)

log_feature(
    "Products",
    "average_product_rating",
    "Average customer rating"
)

summarize_feature(
    products,
    "average_product_rating"
)

validate_feature(
    products,
    "average_product_rating"
)

Validation : average_product_rating


,average_product_rating
0,5.00
1,5.00
2,5.00
3,1.00
4,5.00


count   32789.00
mean        4.05
std         1.21
min         1.00
25%         3.61
50%         4.50
75%         5.00
max         5.00
Name: average_product_rating, dtype: float64

Missing Values : 162
------------------------------------------------------------


In [95]:
# ==========================================
# Feature D6 : Product Review Count
# ==========================================

product_reviews = (
    product_master
    .groupby("product_id", as_index=False)
    .agg(
        product_review_count=("review_score", "count")
    )
)

products = products.merge(
    product_reviews,
    on="product_id",
    how="left"
)

products["product_review_count"] = (
    products["product_review_count"]
    .fillna(0)
    .astype(int)
)

log_feature(
    "Products",
    "product_review_count",
    "Total reviews received"
)

summarize_feature(
    products,
    "product_review_count"
)

validate_feature(
    products,
    "product_review_count"
)

Validation : product_review_count


,product_review_count
0,1
1,1
2,1
3,1
4,1


count   32951.00
mean        3.41
std        10.61
min         0.00
25%         1.00
50%         1.00
75%         3.00
max       524.00
Name: product_review_count, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [96]:
# ==========================================
# Feature D7 : Product Volume
# ==========================================

products["product_volume_cm3"] = (
    products["product_length_cm"] *
    products["product_height_cm"] *
    products["product_width_cm"]
)

log_feature(
    "Products",
    "product_volume_cm3",
    "Calculated package volume in cubic centimeters"
)

summarize_feature(products, "product_volume_cm3")
validate_feature(products, "product_volume_cm3")

Validation : product_volume_cm3


,product_volume_cm3
0,2240.00
1,10800.00
2,2430.00
3,2704.00
4,4420.00


count    32949.00
mean     16564.10
std      27057.04
min        168.00
25%       2880.00
50%       6840.00
75%      18480.00
max     296208.00
Name: product_volume_cm3, dtype: float64

Missing Values : 2
------------------------------------------------------------


In [97]:
# ==========================================
# Feature D8 : Product Density
# ==========================================

products["product_density"] = (
    products["product_weight_g"] /
    products["product_volume_cm3"]
)

products.loc[
    products["product_volume_cm3"] <= 0,
    "product_density"
] = np.nan

log_feature(
    "Products",
    "product_density",
    "Weight per cubic centimeter"
)

summarize_feature(products, "product_density")
validate_feature(products, "product_density")

Validation : product_density


,product_density
0,0.10
1,0.09
2,0.06
3,0.14
4,0.14


count   32949.00
mean        0.20
std         1.01
min         0.00
25%         0.07
50%         0.12
75%         0.20
max        85.23
Name: product_density, dtype: float64

Missing Values : 2
------------------------------------------------------------


In [98]:
# ==========================================
# Feature D9 : Weight Category
# ==========================================

products["weight_category"] = pd.cut(
    products["product_weight_g"],
    bins=[0, 500, 2000, np.inf],
    labels=[
        "Light",
        "Medium",
        "Heavy"
    ],
    include_lowest=True
)

log_feature(
    "Products",
    "weight_category",
    "Product weight classification"
)

summarize_feature(products, "weight_category")

In [99]:
# ==========================================
# Feature D10 : Volume Category
# ==========================================

products["volume_category"] = pd.qcut(
    products["product_volume_cm3"],
    q=4,
    labels=[
        "Small",
        "Medium",
        "Large",
        "Extra Large"
    ]
)

log_feature(
    "Products",
    "volume_category",
    "Volume-based product segmentation"
)

summarize_feature(products, "volume_category")

In [100]:
# ==========================================
# Feature D11 : Freight Ratio
# ==========================================

products["freight_ratio"] = (
    products["average_freight"] /
    products["average_selling_price"]
)

log_feature(
    "Products",
    "freight_ratio",
    "Freight as a proportion of selling price"
)

summarize_feature(products, "freight_ratio")
validate_feature(products, "freight_ratio")

Validation : freight_ratio


,freight_ratio
0,0.68
1,0.07
2,0.10
3,0.08
4,0.22


count   32951.00
mean        0.32
std         0.35
min         0.00
25%         0.13
50%         0.23
75%         0.40
max        23.04
Name: freight_ratio, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [101]:
# ==========================================
# Feature D12 : Product Value Tier
# ==========================================

products["product_value_tier"] = pd.qcut(
    products["product_revenue"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

log_feature(
    "Products",
    "product_value_tier",
    "Revenue-based product segmentation"
)

summarize_feature(products, "product_value_tier")

## Feature D13 — Product Popularity Tier

### Business Purpose

Products are segmented according to their sales volume.

This enables quick identification of:

- Low-demand products
- Moderate-demand products
- High-demand products
- Best-selling products

In [103]:
# ==========================================
# Feature D13 : Product Popularity Tier
# ==========================================

conditions = [
    products["product_sales_count"] == 1,
    products["product_sales_count"].between(2, 5),
    products["product_sales_count"].between(6, 20),
    products["product_sales_count"] > 20
]

choices = [
    "Rarely Sold",
    "Moderately Sold",
    "Popular",
    "Best Seller"
]

products["product_popularity_tier"] = np.select(
    conditions,
    choices,
    default=None
)

log_feature(
    "Products",
    "product_popularity_tier",
    "Popularity based on sales count"
)

summarize_feature(
    products,
    "product_popularity_tier"
)

validate_feature(
    products,
    "product_popularity_tier"
)

Validation : product_popularity_tier


,product_popularity_tier
0,Rarely Sold
1,Rarely Sold
2,Rarely Sold
3,Rarely Sold
4,Rarely Sold


count           32951
unique              4
top       Rarely Sold
freq            18012
Name: product_popularity_tier, dtype: object

Missing Values : 0
------------------------------------------------------------


In [104]:
# ==========================================
# Feature D14 : Product Rating Category
# ==========================================

conditions = [
    products["average_product_rating"] >= 4.5,
    (products["average_product_rating"] >= 4.0) &
    (products["average_product_rating"] < 4.5),
    (products["average_product_rating"] >= 3.0) &
    (products["average_product_rating"] < 4.0),
    products["average_product_rating"] < 3.0
]

choices = [
    "Excellent",
    "Good",
    "Average",
    "Poor"
]

products["product_rating_category"] = np.select(
    conditions,
    choices,
    default=None
)

log_feature(
    "Products",
    "product_rating_category",
    "Rating-based product segmentation"
)

summarize_feature(products, "product_rating_category")

In [105]:
# ==========================================
# Feature D15 : Shipping Cost Category
# ==========================================

products["shipping_cost_category"] = pd.cut(
    products["freight_ratio"],
    bins=[0, 0.10, 0.25, 0.50, np.inf],
    labels=[
        "Low",
        "Moderate",
        "High",
        "Very High"
    ],
    include_lowest=True
)

log_feature(
    "Products",
    "shipping_cost_category",
    "Shipping cost relative to selling price"
)

summarize_feature(products, "shipping_cost_category")

In [106]:
# ==========================================
# Feature D16 : Shipping Efficiency
# ==========================================

products["shipping_efficiency"] = (
    products["average_selling_price"] /
    products["average_freight"]
)

products.loc[
    products["average_freight"] <= 0,
    "shipping_efficiency"
] = np.nan

log_feature(
    "Products",
    "shipping_efficiency",
    "Price generated per unit freight cost"
)

summarize_feature(products, "shipping_efficiency")

validate_feature(products, "shipping_efficiency")

Validation : shipping_efficiency


,shipping_efficiency
0,1.48
1,13.79
2,10.20
3,11.77
4,4.57


count   32951.00
mean        7.75
std        74.06
min         0.04
25%         2.53
50%         4.41
75%         7.96
max     11300.00
Name: shipping_efficiency, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [107]:
product_features = [
    "product_sales_count",
    "product_revenue",
    "average_selling_price",
    "average_freight",
    "average_product_rating",
    "product_review_count",
    "product_volume_cm3",
    "product_density",
    "weight_category",
    "volume_category",
    "freight_ratio",
    "product_value_tier",
    "product_popularity_tier",
    "product_rating_category",
    "shipping_cost_category",
    "shipping_efficiency"
]

summary = pd.DataFrame({
    "Feature": product_features,
    "Datatype": [str(products[f].dtype) for f in product_features],
    "Missing Values": [products[f].isna().sum() for f in product_features],
    "Unique Values": [products[f].nunique() for f in product_features]
})

display(summary)

,Feature,Datatype,Missing Values,Unique Values
0,product_sales_count,int64,0,141
1,product_revenue,float64,0,10310
2,average_selling_price,float64,0,8827
3,average_freight,float64,0,12617
4,average_product_rating,float64,162,649
5,product_review_count,int64,0,142
6,product_volume_cm3,float64,2,4525
7,product_density,float64,2,13659
8,weight_category,category,2,3
9,volume_category,category,2,4


# Section E — Payment Features

## Business Objective

Payment features provide insights into customer payment behavior, installment usage, and transaction characteristics.

These engineered features support:

- Payment behavior analysis
- Customer purchasing patterns
- Installment analysis
- Financial reporting

The engineered dataset will be used during Exploratory Data Analysis (EDA) and Power BI reporting.

In [108]:
# ==========================================
# Feature E1 : Average Installment Value
# ==========================================

payments["average_installment_value"] = (
    payments["payment_value"] /
    payments["payment_installments"]
)

payments.loc[
    payments["payment_installments"] <= 0,
    "average_installment_value"
] = np.nan

log_feature(
    "Payments",
    "average_installment_value",
    "Average amount paid per installment"
)

summarize_feature(payments, "average_installment_value")
validate_feature(payments, "average_installment_value")

Validation : average_installment_value


,average_installment_value
0,12.42
1,24.39
2,65.71
3,13.47
4,64.22


count   103884.00
mean        79.57
std        134.70
min          0.00
25%         26.52
50%         51.33
75%         89.93
max      13664.08
Name: average_installment_value, dtype: float64

Missing Values : 2
------------------------------------------------------------


In [109]:
# ==========================================
# Feature E2 : Installment Category
# ==========================================

conditions = [
    payments["payment_installments"] == 1,
    payments["payment_installments"].between(2,3),
    payments["payment_installments"].between(4,6),
    payments["payment_installments"] >= 7
]

choices = [
    "Single Payment",
    "Low Installments",
    "Medium Installments",
    "High Installments"
]

payments["installment_category"] = np.select(
    conditions,
    choices,
    default=None
)

log_feature(
    "Payments",
    "installment_category",
    "Installment-based payment category"
)

summarize_feature(payments, "installment_category")

In [110]:
# ==========================================
# Feature E3 : High Installment Flag
# ==========================================

payments["high_installment_flag"] = (
    payments["payment_installments"] >= 6
)

log_feature(
    "Payments",
    "high_installment_flag",
    "Indicates payments with six or more installments"
)

summarize_feature(payments, "high_installment_flag")

In [111]:
# ==========================================
# Feature E4 : Payment Value Tier
# ==========================================

payments["payment_value_tier"] = pd.qcut(
    payments["payment_value"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Premium"
    ]
)

log_feature(
    "Payments",
    "payment_value_tier",
    "Payment value segmentation"
)

summarize_feature(payments, "payment_value_tier")

In [112]:
payment_features = [
    "average_installment_value",
    "installment_category",
    "high_installment_flag",
    "payment_value_tier"
]

payment_summary = pd.DataFrame({
    "Feature": payment_features,
    "Datatype": [str(payments[f].dtype) for f in payment_features],
    "Missing Values": [payments[f].isna().sum() for f in payment_features],
    "Unique Values": [payments[f].nunique() for f in payment_features]
})

display(payment_summary)

,Feature,Datatype,Missing Values,Unique Values
0,average_installment_value,float64,2,44773
1,installment_category,str,2,4
2,high_installment_flag,bool,0,2
3,payment_value_tier,category,0,4


In [113]:
# ==========================================
# Feature F1 : Rating Category
# ==========================================

conditions = [
    reviews["review_score"] >= 5,
    reviews["review_score"] == 4,
    reviews["review_score"] == 3,
    reviews["review_score"] <= 2
]

choices = [
    "Excellent",
    "Good",
    "Average",
    "Poor"
]

reviews["rating_category"] = np.select(
    conditions,
    choices,
    default=None
)

log_feature(
    "Reviews",
    "rating_category",
    "Review rating category"
)

summarize_feature(reviews, "rating_category")

In [114]:
# ==========================================
# Feature F2 : Review Sentiment
# ==========================================

conditions = [
    reviews["review_score"] >= 4,
    reviews["review_score"] == 3,
    reviews["review_score"] <= 2
]

choices = [
    "Positive",
    "Neutral",
    "Negative"
]

reviews["review_sentiment"] = np.select(
    conditions,
    choices,
    default=None
)

log_feature(
    "Reviews",
    "review_sentiment",
    "Customer sentiment derived from review score"
)

summarize_feature(reviews, "review_sentiment")

In [115]:
reviews["positive_review"] = reviews["review_score"] >= 4

log_feature(
    "Reviews",
    "positive_review",
    "Positive review indicator"
)

summarize_feature(reviews, "positive_review")

In [116]:
reviews["negative_review"] = reviews["review_score"] <= 2

log_feature(
    "Reviews",
    "negative_review",
    "Negative review indicator"
)

summarize_feature(reviews, "negative_review")

In [117]:
reviews["neutral_review"] = reviews["review_score"] == 3

log_feature(
    "Reviews",
    "neutral_review",
    "Neutral review indicator"
)

summarize_feature(reviews, "neutral_review")

In [118]:
# ==========================================
# Seller Average Order Value (Correct)
# ==========================================

seller_orders = (
    order_items
    .groupby(["seller_id", "order_id"], as_index=False)
    .agg(
        order_value=(
            "price",
            "sum"
        )
    )
)

seller_aov = (
    seller_orders
    .groupby("seller_id", as_index=False)
    .agg(
        seller_avg_order_value=(
            "order_value",
            "mean"
        )
    )
)

# Remove old column if it exists
if "seller_avg_order_value" in sellers.columns:
    sellers.drop(
        columns=["seller_avg_order_value"],
        inplace=True
    )

sellers = sellers.merge(
    seller_aov,
    on="seller_id",
    how="left"
)

log_feature(
    "Sellers",
    "seller_avg_order_value",
    "Average value of orders handled by seller"
)

summarize_feature(
    sellers,
    "seller_avg_order_value"
)

validate_feature(
    sellers,
    "seller_avg_order_value"
)

Validation : seller_avg_order_value


,seller_avg_order_value
0,72.90
1,292.58
2,158.00
3,79.99
4,167.99


count   3095.00
mean     194.65
std      346.49
min        3.50
25%       60.12
50%      105.87
75%      189.97
max     6729.00
Name: seller_avg_order_value, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [119]:
# ==========================================
# Seller Review Count
# ==========================================

seller_review_count = (
    seller_master
    .groupby("seller_id", as_index=False)
    .agg(
        seller_review_count=(
            "review_score",
            "count"
        )
    )
)

sellers = sellers.merge(
    seller_review_count,
    on="seller_id",
    how="left"
)

sellers["seller_review_count"] = (
    sellers["seller_review_count"]
    .fillna(0)
    .astype(int)
)

log_feature(
    "Sellers",
    "seller_review_count",
    "Total reviews received by seller"
)

summarize_feature(
    sellers,
    "seller_review_count"
)

validate_feature(
    sellers,
    "seller_review_count"
)

Validation : seller_review_count


,seller_review_count
0,3
1,41
2,1
3,1
4,1


count   3095.00
mean      36.31
std      119.19
min        0.00
25%        2.00
50%        7.00
75%       24.00
max     2020.00
Name: seller_review_count, dtype: float64

Missing Values : 0
------------------------------------------------------------


In [120]:
# ==========================================
# Remove Redundant Feature
# ==========================================

if "shipping_efficiency" in products.columns:
    products.drop(
        columns=["shipping_efficiency"],
        inplace=True
    )

print("✓ shipping_efficiency removed.")

✓ shipping_efficiency removed.


In [121]:
# ==========================================
# Final Feature Summary
# ==========================================

datasets = {
    "Orders": orders,
    "Customers": customers,
    "Sellers": sellers,
    "Products": products,
    "Payments": payments,
    "Reviews": reviews
}

feature_summary = []

for dataset_name, df in datasets.items():

    for col in df.columns:

        feature_summary.append({
            "Dataset": dataset_name,
            "Feature": col,
            "Data Type": str(df[col].dtype),
            "Missing Values": df[col].isna().sum(),
            "Unique Values": df[col].nunique()
        })

feature_summary = pd.DataFrame(feature_summary)

display(feature_summary.head())

,Dataset,Feature,Data Type,Missing Values,Unique Values
0,Orders,order_id,str,0,99441
1,Orders,customer_id,str,0,99441
2,Orders,order_status,str,0,8
3,Orders,order_purchase_timestamp,datetime64[us],0,98875
4,Orders,order_approved_at,datetime64[us],160,90733


In [122]:
# ==========================================
# Export Engineered Datasets
# ==========================================

from pathlib import Path

output_dir = Path("../data/featured")
output_dir.mkdir(parents=True, exist_ok=True)

customers.to_csv(output_dir / "customers.csv", index=False)
orders.to_csv(output_dir / "orders.csv", index=False)
order_items.to_csv(output_dir / "order_items.csv", index=False)
payments.to_csv(output_dir / "payments.csv", index=False)
reviews.to_csv(output_dir / "reviews.csv", index=False)
products.to_csv(output_dir / "products.csv", index=False)
sellers.to_csv(output_dir / "sellers.csv", index=False)
geolocation.to_csv(output_dir / "geolocation.csv", index=False)
category_translation.to_csv(
    output_dir / "product_category_translation.csv",
    index=False
)

print("✓ All featured datasets exported successfully.")

✓ All featured datasets exported successfully.


In [124]:
# ==========================================
# Export Reports
# ==========================================

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# Convert feature log to DataFrame
feature_log_df = pd.DataFrame(feature_log)

feature_log_df.to_csv(
    reports_dir / "feature_engineering_log.csv",
    index=False
)

feature_summary.to_csv(
    reports_dir / "feature_summary.csv",
    index=False
)

print("✓ Feature engineering reports exported.")

✓ Feature engineering reports exported.


In [125]:
from pathlib import Path

featured = Path("../data/featured")
reports = Path("../reports")

print("Featured Files")
print("-"*40)

for f in sorted(featured.glob("*")):
    print(f.name)

print()

print("Reports")
print("-"*40)

for f in sorted(reports.glob("*")):
    print(f.name)

Featured Files
----------------------------------------
customers.csv
geolocation.csv
order_items.csv
orders.csv
payments.csv
product_category_translation.csv
products.csv
reviews.csv
sellers.csv

Reports
----------------------------------------
cleaning_log.csv
cleaning_summary.csv
feature_engineering_log.csv
feature_summary.csv
